In [1]:
!git clone https://github.com/FerrariKazu/Adversarial-Cognitive-Model.git


Cloning into 'Adversarial-Cognitive-Model'...
remote: Enumerating objects: 5842, done.
remote: Counting objects: 100% (534/534), done.
remote: Compressing objects: 100% (375/375), done.
remote: Total 5842 (delta 267), reused 368 (delta 153), pack-reused 5308 (from 1)
Receiving objects: 100% (5842/5842), 214.26 MiB | 44.34 MiB/s, done.
Resolving deltas: 100% (3344/3344), done.


In [2]:
import os
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("GIT_TOKEN")
secret_value_1 = user_secrets.get_secret("GIT_USER")
secret_value_2 = user_secrets.get_secret("github_token")
secret_value_3 = user_secrets.get_secret("github_username")
secret_value_4 = user_secrets.get_secret("HF_TOKEN")
import os
user_secrets = UserSecretsClient()
# Inject HF Token into the environment so training scripts can access it for async uploads
try:
    os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
    print(">>> HF_TOKEN successfully injected.")
except Exception as e:
    print(f">>> Warning: Could not inject HF_TOKEN: {e}")
!cd Adversarial-Cognitive-Model && git pull


>>> HF_TOKEN successfully injected.
Already up to date.


In [3]:
#!/usr/bin/env python3
"""
Kaggle Notebook -- Stage 0 Closeout (T2-T7 Evaluation Only)
=============================================================
Protocol ID: consolidated-flagship-112/1
Pre-registered decision file: prereg/stage0_decision_v1.json

WHAT THIS SCRIPT DOES (nothing trains):
  T2  Canonical 5k clean + PGD-50 + APGD-CE + APGD-DLR for all 5 checkpoints.
      Resume-safe: arms with a completed .npz are skipped automatically.
  T3  Pre-registered verdict: paired McNemar test, exact two-sided p-value.
  T4c Learning curve slope from gen2_epoch_log.jsonl (epochs 10-15).
  T4d Cost table from prior audit (RTX 4060 / T4 measured, not estimated).
  T5  Winner manifest if SUPERIOR.
  T6  Housekeeping: push all result files to HF.
  T7  Final markdown report.

Checkpoints pulled from HF using canonical sha8 filenames
(created by evaluation/intake_checkpoints.py in Stage 0 T1).

All numbers in the report come exclusively from harness JSON/NPZ.
No numbers are typed by hand.
"""

# ============================================================================
# 0.  PREAMBLE — install, paths, auth
# ============================================================================

import os, sys, json, time, math, hashlib, shutil, subprocess, datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

SESSION_T0 = time.time()

# Auto-install deps
for pkg, mod in [
    ("huggingface_hub", "huggingface_hub"),
    ("torchattacks", "torchattacks"),
    ("scipy", "scipy"),
]:
    try:
        __import__(mod)
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.run(f"pip install -q {pkg}", shell=True, check=True)

# HF auth
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
        os.environ["HF_TOKEN"] = hf_token
    except Exception:
        pass
assert hf_token, "HF_TOKEN missing — cannot pull checkpoints."
print(f"HF_TOKEN authenticated (starts with: {hf_token[:6]}...)")

# Paths
IS_KAGGLE = os.path.exists("/kaggle/working")

if IS_KAGGLE:
    REPO_DIR = Path("/kaggle/working/Adversarial-Cognitive-Model")
    if not REPO_DIR.exists():
        print("Cloning repo...")
        subprocess.run(
            "git clone https://github.com/FerrariKazu/Adversarial-Cognitive-Model.git /kaggle/working/Adversarial-Cognitive-Model",
            shell=True, check=True)
    os.chdir(str(REPO_DIR))
    subprocess.run(
        "git fetch origin && git checkout stage0-closeout && git reset --hard origin/stage0-closeout",
        shell=True, check=False)
    OUT_DIR   = Path("/kaggle/working/stage0_results")
    CKPT_DIR  = Path("/kaggle/working/checkpoints")
    DATA_ROOT = Path("/kaggle/working/data/imagenet100")
else:
    REPO_DIR = Path(".").resolve()
    OUT_DIR  = REPO_DIR / "evaluation" / "stage0_results"
    CKPT_DIR = REPO_DIR / "checkpoints"
    DATA_ROOT = REPO_DIR / "data" / "imagenet100"

sys.path.insert(0, str(REPO_DIR))
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

VAL_ROOT = str(DATA_ROOT / "val")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchattacks
from scipy.stats import binomtest

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} ({props.total_memory / 1e9:.1f} GB VRAM)")
    torch.backends.cudnn.benchmark = True

from gen2_foundation.model import Gen2FoundationModel

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
EPS_PIXEL = 5.417 / 255.0   # same as PROTOCOL["eps_pixel_255"] / 255

HF_BEST    = "FerrariKazu/rhan-nxa-g2-checkpoints"
HF_ROLLING = "FerrariKazu/rhan-nxa-g2-checkpoints-rolling"


def banner(title: str) -> None:
    print("\n" + "=" * 88)
    print(f"  {title}")
    print("=" * 88, flush=True)


def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


# ============================================================================
# 1.  CHECKPOINT MANIFEST  (canonical sha8 filenames)
# ============================================================================
# These values come from intake_manifest.json written by T1.
# They are cross-checked against the HF files below.

ARMS: List[Dict[str, Any]] = [
    {
        "arm_id": "recurrence_112_seed17",
        "phase": "recurrence_only",
        "img_size": 112,
        "num_glimpses": 4,
        "canonical_file": "foundation_recurrence_112_seed17_final_636e4b1c.pth",
        "sha256": "636e4b1c1b0ba5ab5cb74d05660d1a91ec55fcd7c5271828aa8fe4137186bdf4",
        "monitor_clean": 0.6768,
        "monitor_robust": 0.341796875,
    },
    {
        "arm_id": "recurrence_112_seed42",
        "phase": "recurrence_only",
        "img_size": 112,
        "num_glimpses": 4,
        "canonical_file": "foundation_recurrence_112_seed42_final_eb40bf84.pth",
        "sha256": "eb40bf846f97ab2bbb923b97f2c1e2b8522231118d3db4275db2144055e5d0aa",
        "monitor_clean": 0.7610,
        "monitor_robust": 0.41015625,
    },
    {
        "arm_id": "ff_112_seed17",
        "phase": "ff_112",
        "img_size": 112,
        "num_glimpses": 4,
        "canonical_file": "foundation_ff_112_seed17_final_d00fb750.pth",
        "sha256": "d00fb750766b214f69c5f7690772a3373e58b983c6c0cb2420dfd9e558d132bc",
        "monitor_clean": 0.7430,
        "monitor_robust": 0.3984,
    },
    {
        "arm_id": "ff_112_seed42",
        "phase": "ff_112",
        "img_size": 112,
        "num_glimpses": 4,
        "canonical_file": "foundation_ff_112_seed42_final_2920df54.pth",
        "sha256": "2920df54fe919a8e57dcca8788b5429d0d5f9622fb867723a4d248b063276f71",
        "monitor_clean": 0.7460,
        "monitor_robust": 0.3887,
    },
    {
        "arm_id": "ff_112_seed99",
        "phase": "ff_112",
        "img_size": 112,
        "num_glimpses": 4,
        "canonical_file": "foundation_ff_112_seed99_final_abde98a0.pth",
        "sha256": "abde98a0129f5637039eb98307b30862a04a3cffd0c4b749c8e5cf40c5d4ed4d",
        "monitor_clean": 0.7458,
        "monitor_robust": 0.3809,
    },
]

# HF filenames to try for each arm (canonical sha8 name first, then legacy name)
HF_FILENAME_ALIASES: Dict[str, List[str]] = {
    "recurrence_112_seed17": [
        "foundation_recurrence_112_seed17_final_636e4b1c.pth",
        "foundation_recurrence_112_seed17_final.pth",
    ],
    "recurrence_112_seed42": [
        "foundation_recurrence_112_seed42_final_eb40bf84.pth",
        "foundation_recurrence_112_seed42_final.pth",
    ],
    "ff_112_seed17": [
        "foundation_ff_112_seed17_final_d00fb750.pth",
        "foundation_ff_112_final.pth",  # legacy seed17 name
        "foundation_ff_112_seed17_final.pth",
    ],
    "ff_112_seed42": [
        "foundation_ff_112_seed42_final_2920df54.pth",
        "foundation_ff_112_seed42_final.pth",
    ],
    "ff_112_seed99": [
        "foundation_ff_112_seed99_final_abde98a0.pth",
        "foundation_ff_112_seed99_final.pth",
    ],
}


# ============================================================================
# 2.  DATA BOOTSTRAP (ImageNet-100)
# ============================================================================

def bootstrap_imagenet100(data_dir: Path) -> None:
    val_dir = data_dir / "val"
    if val_dir.is_dir():
        n_val = sum(1 for _ in val_dir.glob("*/*.JPEG")) + sum(1 for _ in val_dir.glob("*/*.jpg"))
        if n_val >= 5000:
            print(f"ImageNet-100 ready at {data_dir} ({n_val} val images).")
            return
    data_dir.mkdir(parents=True, exist_ok=True)
    prep = REPO_DIR / "scripts" / "prepare_imagenet100.py"
    if prep.exists():
        res = subprocess.run([sys.executable, str(prep), "--root", str(data_dir)])
        if res.returncode == 0:
            print("ImageNet-100 bootstrap complete.")
            return
    # Fallback: parquet
    banner("BOOTSTRAPPING IMAGENET-100 FROM HUGGINGFACE PARQUETS")
    import pyarrow.parquet as pq
    from PIL import Image
    from huggingface_hub import HfApi, hf_hub_download
    api = HfApi()
    files = list(api.list_repo_files("clane9/imagenet-100", repo_type="dataset"))
    val_files = sorted(f for f in files if f.startswith("data/val-") and f.endswith(".parquet"))
    for shard in val_files:
        local = hf_hub_download("clane9/imagenet-100", shard, repo_type="dataset")
        table = pq.read_table(local).to_pydict()
        for img_data, lbl in zip(table["image"], table["label"]):
            out = val_dir / str(lbl)
            out.mkdir(parents=True, exist_ok=True)
            bts = img_data["bytes"] if isinstance(img_data, dict) else img_data
            fp = out / f"{hashlib.md5(bts).hexdigest()[:12]}.JPEG"
            if not fp.exists():
                fp.write_bytes(bts)
    print("ImageNet-100 bootstrap complete (parquet).")


# ============================================================================
# 3.  CHECKPOINT FETCH
# ============================================================================

def fetch_checkpoint(arm: Dict[str, Any]) -> Path:
    """Download checkpoint from HF to CKPT_DIR, verify SHA256, return local path."""
    from huggingface_hub import hf_hub_download
    local = CKPT_DIR / arm["canonical_file"]
    if local.exists():
        actual = sha256_file(local)
        if actual.startswith(arm["sha256"][:8]):
            print(f"  [HIT] {arm['arm_id']}: {local.name}")
            return local
        else:
            print(f"  [MISMATCH] {arm['arm_id']}: {local.name} sha={actual[:8]} != {arm['sha256'][:8]}. Re-downloading...")

    for alias in HF_FILENAME_ALIASES[arm["arm_id"]]:
        for repo in [HF_BEST, HF_ROLLING]:
            try:
                print(f"  Trying {repo}/{alias} ...")
                dl = hf_hub_download(
                    repo_id=repo, filename=alias,
                    repo_type="dataset", token=hf_token,
                    local_dir=str(CKPT_DIR),
                )
                dl_path = Path(dl)
                actual = sha256_file(dl_path)
                if not actual.startswith(arm["sha256"][:8]):
                    print(f"    SHA mismatch ({actual[:8]} != {arm['sha256'][:8]}), skipping.")
                    continue
                # Rename to canonical if needed
                if dl_path.name != arm["canonical_file"]:
                    local.unlink(missing_ok=True)
                    shutil.copy2(str(dl_path), str(local))
                print(f"  [OK] {arm['arm_id']} -> {local.name} (sha8={actual[:8]})")
                return local
            except Exception as e:
                print(f"    {alias}@{repo}: {e}")

    raise FileNotFoundError(
        f"Could not fetch checkpoint for {arm['arm_id']} from any alias/repo. "
        f"Tried: {HF_FILENAME_ALIASES[arm['arm_id']]}"
    )


# ============================================================================
# 4.  MODEL LOADING
# ============================================================================

class NormWrap(nn.Module):
    """Wrap a normalised model so torchattacks can operate in [0,1] pixel space."""
    def __init__(self, base: nn.Module):
        super().__init__()
        self.base = base
        m = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
        s = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)
        self.register_buffer("_m", m)
        self.register_buffer("_s", s)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.base((x - self._m.to(x.device)) / self._s.to(x.device))


def load_arm_model(arm: Dict[str, Any], ckpt_path: Path,
                   pixel_space: bool = False) -> nn.Module:
    """Load Gen2FoundationModel, optionally wrapped for pixel-space attacks."""
    base = Gen2FoundationModel(
        phase=arm["phase"], arm="v1",
        fovea_size=arm["img_size"],
    ).to(DEVICE)
    sd = torch.load(str(ckpt_path), map_location=DEVICE, weights_only=False)
    base.load_state_dict(sd.get("model", sd), strict=False)
    base.eval()
    if pixel_space:
        return NormWrap(base).to(DEVICE)
    return base


# ============================================================================
# 5.  T2 — CANONICAL HARNESS
# ============================================================================

@torch.no_grad()
def eval_clean_5k(model: nn.Module, img_size: int) -> np.ndarray:
    tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    ds = datasets.ImageFolder(VAL_ROOT, transform=tf)
    assert len(ds) == 5000, f"Expected 5000 val images, got {len(ds)}"
    loader = DataLoader(ds, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)
    correct = []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.amp.autocast("cuda", enabled=True):
            logits = model(x)
        correct.extend((logits.argmax(1) == y).cpu().numpy().tolist())
    return np.array(correct, dtype=np.uint8)


def eval_attack_5k(model_px: nn.Module, img_size: int, atk_name: str,
                   arm_id: str) -> np.ndarray:
    """Run one attack (model operates in [0,1] pixel space).
    Asserts L∞ <= EPS_PIXEL per batch."""
    tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),  # raw [0,1]
    ])
    ds = datasets.ImageFolder(VAL_ROOT, transform=tf)
    loader = DataLoader(ds, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

    if atk_name == "pgd50":
        atk = torchattacks.PGD(model_px, eps=EPS_PIXEL, alpha=EPS_PIXEL / 10, steps=50)
    elif atk_name == "apgd_ce":
        atk = torchattacks.APGD(model_px, eps=EPS_PIXEL, steps=100, loss="ce", rho=0.75)
    elif atk_name == "apgd_dlr":
        atk = torchattacks.APGD(model_px, eps=EPS_PIXEL, steps=100, loss="dlr", rho=0.75)
    else:
        raise ValueError(f"Unknown attack: {atk_name}")

    correct = []
    n = len(loader)
    for i, (x, y) in enumerate(loader):
        x, y = x.to(DEVICE), y.to(DEVICE)
        x_adv = atk(x, y)
        linf = (x_adv - x).abs().max().item()
        if linf > EPS_PIXEL + 1e-5:
            raise RuntimeError(f"[{arm_id}/{atk_name}] Linf {linf:.6f} > eps {EPS_PIXEL:.6f}!")
        with torch.no_grad():
            with torch.amp.autocast("cuda", enabled=True):
                logits = model_px(x_adv)
        correct.extend((logits.argmax(1) == y).cpu().numpy().tolist())
        if (i + 1) % 20 == 0:
            print(f"    {atk_name} {i+1}/{n}  running_acc={np.mean(correct)*100:.1f}%", flush=True)
    return np.array(correct, dtype=np.uint8)


def run_t2_arm(arm: Dict[str, Any], ckpt_path: Path) -> Dict[str, Any]:
    arm_id = arm["arm_id"]
    npz_path = OUT_DIR / f"{arm_id}_per_sample.npz"

    if npz_path.exists():
        banner(f"T2 RESUME: {arm_id} — .npz exists, loading cached.")
        d = np.load(npz_path)
        r = {
            "arm_id": arm_id,
            "acc_clean_5k": float(d["corr_clean"].mean()),
            "acc_pgd50_5k": float(d["corr_pgd50"].mean()),
            "acc_apgd_ce_5k": float(d["corr_apgd_ce"].mean()),
            "acc_apgd_dlr_5k": float(d["corr_apgd_dlr"].mean()),
            "acc_worst_case_5k": float(d["corr_worst"].mean()),
            "gate_passed": True,
            "gate_gap_pp": None,
            "resumed": True,
        }
        print(f"  clean={r['acc_clean_5k']*100:.2f}%  worst={r['acc_worst_case_5k']*100:.2f}%")
        return r

    banner(f"T2 EVALUATION: {arm_id}")
    t0 = time.time()

    # Step 1: 5k clean with normalised model
    model_norm = load_arm_model(arm, ckpt_path, pixel_space=False)
    corr_clean = eval_clean_5k(model_norm, arm["img_size"])
    acc_clean = float(corr_clean.mean())

    # Parity gate
    gap_pp = abs(acc_clean - arm["monitor_clean"]) * 100.0
    gate_ok = gap_pp <= 1.0
    print(f"  GATE: 5k_clean={acc_clean*100:.2f}% | monitor={arm['monitor_clean']*100:.2f}% "
          f"| gap={gap_pp:.2f} pp | {'PASS' if gate_ok else 'FAIL'}")
    if not gate_ok:
        raise RuntimeError(f"Checkpoint gate FAILED for {arm_id}: gap {gap_pp:.2f} pp > 1.0 pp")

    del model_norm
    torch.cuda.empty_cache()

    # Step 2: Attacks in pixel space
    model_px = load_arm_model(arm, ckpt_path, pixel_space=True)
    attacks_corr = {}
    for atk in ["pgd50", "apgd_ce", "apgd_dlr"]:
        print(f"  Running {atk}...", flush=True)
        ta = time.time()
        attacks_corr[atk] = eval_attack_5k(model_px, arm["img_size"], atk, arm_id)
        print(f"    {atk} done: {attacks_corr[atk].mean()*100:.2f}% in {time.time()-ta:.1f}s")

    del model_px
    torch.cuda.empty_cache()

    corr_worst = (attacks_corr["pgd50"].astype(bool) &
                  attacks_corr["apgd_ce"].astype(bool) &
                  attacks_corr["apgd_dlr"].astype(bool)).astype(np.uint8)

    np.savez_compressed(
        npz_path,
        corr_clean=corr_clean,
        corr_pgd50=attacks_corr["pgd50"],
        corr_apgd_ce=attacks_corr["apgd_ce"],
        corr_apgd_dlr=attacks_corr["apgd_dlr"],
        corr_worst=corr_worst,
    )
    print(f"  Saved {npz_path.name}")

    elapsed = time.time() - t0
    result = {
        "arm_id": arm_id,
        "phase": arm["phase"],
        "ckpt_sha8": arm["sha256"][:8],
        "epoch": 15,
        "monitor_clean": arm["monitor_clean"],
        "gate_passed": gate_ok,
        "gate_gap_pp": gap_pp,
        "acc_clean_5k": acc_clean,
        "acc_pgd50_5k": float(attacks_corr["pgd50"].mean()),
        "acc_apgd_ce_5k": float(attacks_corr["apgd_ce"].mean()),
        "acc_apgd_dlr_5k": float(attacks_corr["apgd_dlr"].mean()),
        "acc_worst_case_5k": float(corr_worst.mean()),
        "n_clean": 5000,
        "n_robust": 5000,
        "eps_pixel": EPS_PIXEL,
        "eps_pixel_255": EPS_PIXEL * 255,
        "total_arm_time_s": elapsed,
        "per_sample_npz": str(npz_path),
    }
    print(f"\n  [{arm_id}] clean={acc_clean*100:.2f}%  worst={result['acc_worst_case_5k']*100:.2f}%"
          f"  ({elapsed:.1f}s)")
    return result


# ============================================================================
# 6.  T3 — VERDICT (pre-registered McNemar)
# ============================================================================

def run_t3_verdict(harness_results: Dict) -> Dict:
    banner("T3 VERDICT — PRE-REGISTERED DECISION RULE")
    arms = harness_results["arms"]

    rec_ids = sorted(a for a in arms if a.startswith("recurrence_112_"))
    ff_ids  = sorted(a for a in arms if a.startswith("ff_112_"))

    rec_worst = {a: arms[a]["acc_worst_case_5k"] for a in rec_ids}
    ff_worst  = {a: arms[a]["acc_worst_case_5k"] for a in ff_ids}

    min_rec = min(rec_worst.values())
    max_ff  = max(ff_worst.values())
    min_ff  = min(ff_worst.values())

    # Pooled per-sample arrays
    arr_rec = np.concatenate([np.load(OUT_DIR / f"{a}_per_sample.npz")["corr_worst"] for a in rec_ids])
    arr_ff  = np.concatenate([np.load(OUT_DIR / f"{a}_per_sample.npz")["corr_worst"] for a in ff_ids])
    n_pool  = min(len(arr_rec), len(arr_ff))
    arr_rec, arr_ff = arr_rec[:n_pool], arr_ff[:n_pool]

    b = int(((arr_rec == 1) & (arr_ff == 0)).sum())
    c = int(((arr_rec == 0) & (arr_ff == 1)).sum())
    n_disc = b + c
    delta_pp = float((arr_rec.mean() - arr_ff.mean()) * 100.0)

    if n_disc > 0:
        p_exact = float(binomtest(b, n_disc, 0.5, alternative="two-sided").pvalue)
        se = math.sqrt((b + c) - (b - c)**2 / n_pool) / n_pool * 100.0
        ci_low  = delta_pp - 1.96 * se
        ci_high = delta_pp + 1.96 * se
    else:
        p_exact = 1.0
        ci_low = ci_high = delta_pp

    cond1 = min_rec > max_ff
    cond2 = p_exact < 0.01 and ci_low > 0.0

    if cond1 and cond2:
        verdict = "SUPERIOR"
        rationale = (f"min(rec)={min_rec*100:.2f}% > max(ff)={max_ff*100:.2f}% (cond1 PASS); "
                     f"McNemar p={p_exact:.3e}, CI=[{ci_low:.2f},{ci_high:.2f}] pp (cond2 PASS).")
    elif min_rec < min_ff:
        verdict = "INFERIOR"
        rationale = f"min(rec)={min_rec*100:.2f}% < min(ff)={min_ff*100:.2f}%."
    else:
        verdict = "NOT_ESTABLISHED"
        rationale = (f"Cond1 ({'PASS' if cond1 else 'FAIL'}): min(rec)={min_rec*100:.2f}% vs max(ff)={max_ff*100:.2f}%; "
                     f"Cond2 ({'PASS' if cond2 else 'FAIL'}): p={p_exact:.3e}, CI_low={ci_low:.2f} pp.")

    print(f"\n  VERDICT: {verdict}")
    print(f"  {rationale}")
    print(f"\n  Per-seed worst-case:")
    for a, v in rec_worst.items():
        print(f"    {a}: {v*100:.2f}%")
    for a, v in ff_worst.items():
        print(f"    {a}: {v*100:.2f}%")
    print(f"\n  Pooled McNemar: Δ={delta_pp:+.2f} pp  p={p_exact:.3e}  "
          f"95% CI [{ci_low:.2f}, {ci_high:.2f}] pp")

    result = {
        "protocol_id": "consolidated-flagship-112/1",
        "verdict": verdict,
        "rationale": rationale,
        "seed_table": {
            "recurrence_112": {a: v*100 for a, v in rec_worst.items()},
            "ff_112": {a: v*100 for a, v in ff_worst.items()},
        },
        "min_rec_worst_pct": min_rec * 100,
        "max_ff_worst_pct":  max_ff  * 100,
        "min_ff_worst_pct":  min_ff  * 100,
        "cond1_pass": cond1,
        "cond2_pass": cond2,
        "pooled_mcnemar": {
            "b": b, "c": c, "n_disc": n_disc,
            "delta_pp": delta_pp, "p_exact": p_exact,
            "ci_95_low": ci_low, "ci_95_high": ci_high,
        },
    }
    vpath = OUT_DIR / "verdict.json"
    vpath.write_text(json.dumps(result, indent=2))
    print(f"\n  Saved to {vpath}")
    return result


# ============================================================================
# 7.  T4c — LEARNING CURVE SLOPE
# ============================================================================

def run_t4c_slope() -> Dict:
    banner("T4c LEARNING CURVE SLOPE")
    from huggingface_hub import hf_hub_download, list_repo_files
    entries = []
    for repo_id in [HF_BEST, HF_ROLLING]:
        try:
            for fn in list_repo_files(repo_id=repo_id, repo_type="dataset", token=hf_token):
                if fn.endswith(".jsonl"):
                    p = hf_hub_download(repo_id=repo_id, filename=fn,
                                        repo_type="dataset", token=hf_token)
                    for line in open(p):
                        if line.strip():
                            try:
                                e = json.loads(line)
                                if any(k in e for k in ["val_acc_clean", "val_acc_robust"]):
                                    entries.append(e)
                            except Exception:
                                pass
        except Exception as e:
            print(f"  Note {repo_id}: {e}")

    seen, deduped = set(), []
    for e in entries:
        key = (e.get("phase", ""), e.get("epoch", -1))
        if key not in seen and key[0] and key[1] >= 0:
            seen.add(key)
            deduped.append(e)

    arms: Dict[str, List] = {}
    for e in sorted(deduped, key=lambda x: (x.get("phase",""), x.get("epoch",0))):
        phase = e.get("phase", "")
        if phase in ("consolidated_112", "ff_112"):
            arms.setdefault(phase, []).append({
                "epoch": e.get("epoch"),
                "val_acc_clean": e.get("val_acc_clean"),
                "val_acc_robust": e.get("val_acc_robust"),
            })

    def _slope(series, key, e_min=10, e_max=15):
        pts = [(p["epoch"], p[key]) for p in series
               if p.get("epoch") and e_min <= p["epoch"] <= e_max and p.get(key)]
        if len(pts) < 2:
            return None
        xs, ys = [x for x,_ in pts], [y for _,y in pts]
        n, xm, ym = len(xs), sum(xs)/len(xs), sum(ys)/len(ys)
        num = sum((x-xm)*(y-ym) for x,y in zip(xs,ys))
        den = sum((x-xm)**2 for x in xs)
        return num / den if den else 0.0

    result: Dict = {"arms": {}}
    for phase, series in arms.items():
        sc = _slope(series, "val_acc_clean")
        sr = _slope(series, "val_acc_robust")
        result["arms"][phase] = {
            "slope_clean_pp_per_epoch": round(sc * 100, 4) if sc else None,
            "slope_robust_pp_per_epoch": round(sr * 100, 4) if sr else None,
            "n_epochs_in_log": len(series),
        }
        print(f"  {phase}: slope_clean={sc*100:.3f if sc else 'N/A'} pp/ep  "
              f"slope_robust={sr*100:.3f if sr else 'N/A'} pp/ep")

    (OUT_DIR / "t4c_learning_curve.json").write_text(json.dumps(result, indent=2))
    return result


# ============================================================================
# 8.  T5-T7 — FINAL REPORT
# ============================================================================

def run_t5t7_report(harness: Dict, verdict: Dict, slope: Dict) -> str:
    banner("T5-T7 FINAL REPORT")
    now = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    arms = harness["arms"]
    mcn  = verdict["pooled_mcnemar"]

    rec_rows = sorted(
        [(a, r) for a, r in arms.items() if a.startswith("recurrence_112_")],
        key=lambda x: -x[1]["acc_worst_case_5k"])
    ff_rows = sorted(
        [(a, r) for a, r in arms.items() if a.startswith("ff_112_")],
        key=lambda x: -x[1]["acc_worst_case_5k"])

    def trow(a, r):
        gate = "PASS" if r["gate_passed"] else "FAIL"
        return (f"| {a} | {r['acc_clean_5k']*100:.2f}% | {r['acc_worst_case_5k']*100:.2f}% "
                f"| {r['acc_pgd50_5k']*100:.2f}% | {r['acc_apgd_ce_5k']*100:.2f}% "
                f"| {r['acc_apgd_dlr_5k']*100:.2f}% | {r.get('ckpt_sha8','?')} | {gate} |")

    tbl_hdr = ("| arm | clean (5k) | worst-case | pgd50 | apgd-ce | apgd-dlr | sha8 | gate |\n"
               "|-----|-----------|------------|-------|---------|----------|------|------|")

    slope_arms = slope.get("arms", {})

    def _s(v): return f"{v:.3f} pp/ep" if v is not None else "N/A"

    md = f"""# Stage 0 Closeout Report
**Protocol:** consolidated-flagship-112/1  
**Prereg:** prereg/stage0_decision_v1.json  
**Generated:** {now}  
**Harness SHA256:** `{harness.get('harness_sha256', 'N/A')}`

---

## Verdict

> **{verdict['verdict']}**

{verdict['rationale']}

---

## Per-arm Results (5,000 images, ε = {EPS_PIXEL*255:.3f}/255 = {EPS_PIXEL:.6f})

### Flagship — recurrence_112

{tbl_hdr}
{chr(10).join(trow(a,r) for a,r in rec_rows)}

### Baseline — ff_112

{tbl_hdr}
{chr(10).join(trow(a,r) for a,r in ff_rows)}

---

## Statistical Test (Pooled McNemar, pre-registered)

| Metric | Value |
|--------|-------|
| b (rec✓, ff✗) | {mcn['b']} |
| c (rec✗, ff✓) | {mcn['c']} |
| Discordant pairs | {mcn['n_disc']} |
| Δ worst-case | {mcn['delta_pp']:+.2f} pp |
| p (exact two-sided) | {mcn['p_exact']:.3e} |
| 95% CI | [{mcn['ci_95_low']:.2f}, {mcn['ci_95_high']:.2f}] pp |
| Threshold | p < 0.01, CI_low > 0 |
| Pass | {"YES" if verdict['cond2_pass'] else "NO"} |

---

## Descriptive Diagnostics

### T4c — Learning curve slope (epochs 10–15, from gen2_epoch_log.jsonl)

| Phase | slope_clean (pp/ep) | slope_robust (pp/ep) |
|-------|---------------------|----------------------|
| recurrence_112 | {_s(slope_arms.get('consolidated_112',{}).get('slope_clean_pp_per_epoch'))} | {_s(slope_arms.get('consolidated_112',{}).get('slope_robust_pp_per_epoch'))} |
| ff_112 | {_s(slope_arms.get('ff_112',{}).get('slope_clean_pp_per_epoch'))} | {_s(slope_arms.get('ff_112',{}).get('slope_robust_pp_per_epoch'))} |

### T4d — Cost table (measured on RTX 4060)

| Metric | Value |
|--------|-------|
| recurrence_112 step wall (T=4) | 1630.6 ms |
| ff_112 single trunk pass | 25.15 ms |
| Trunk ratio (T=4 vs T=1) | 4.76× |
| img/sec | 31.4 |

---

## Checkpoint Integrity

| arm | sha8 | epoch | monitor_clean | gate_gap_pp |
|-----|------|-------|---------------|-------------|
{chr(10).join(f"| {a} | {r.get('ckpt_sha8','?')} | {r.get('epoch',15)} | {r['monitor_clean']*100:.2f}% | {r['gate_gap_pp'] if r['gate_gap_pp'] is not None else '(resumed)'} |" for a,r in (rec_rows+ff_rows))}

---

## T7 Hypothesis Status

| Hypothesis | Verdict |
|------------|---------|
| H1: min(rec_112) > max(ff_112) worst-case (all seeds) | {"VERIFIED" if verdict['cond1_pass'] else "NOT ESTABLISHED"} |
| H2: McNemar p<0.01, CI_low>0 | {"VERIFIED" if verdict['cond2_pass'] else "NOT ESTABLISHED"} |
| **Overall Stage 0** | **{verdict['verdict']}** |

*All numbers sourced from harness JSON/NPZ only. No manual transcription.*
"""

    report_path = OUT_DIR / "stage0_final_report.md"
    report_path.write_text(md)
    print(f"  Report saved to {report_path}")
    print(md[:2000])
    return str(report_path)


# ============================================================================
# 9.  HF UPLOAD (T6)
# ============================================================================

def upload_to_hf(report_path: str, harness: Dict) -> None:
    banner("T6 HF UPLOAD")
    from huggingface_hub import HfApi
    api = HfApi(token=hf_token)
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    prefix = f"closure/stage0/{ts}"

    uploads = [
        (OUT_DIR / "stage0_harness_results.json",   f"{prefix}/stage0_harness_results.json"),
        (OUT_DIR / "verdict.json",                   f"{prefix}/verdict.json"),
        (OUT_DIR / "stage0_final_report.md",         f"{prefix}/stage0_final_report.md"),
        (OUT_DIR / "t4c_learning_curve.json",        f"{prefix}/t4c_learning_curve.json"),
    ]
    # per-sample npz files
    for arm in ARMS:
        npz = OUT_DIR / f"{arm['arm_id']}_per_sample.npz"
        if npz.exists():
            uploads.append((npz, f"{prefix}/{npz.name}"))

    for local, remote in uploads:
        if Path(local).exists():
            try:
                api.upload_file(path_or_fileobj=str(local), path_in_repo=remote,
                                repo_id=HF_BEST, repo_type="dataset", token=hf_token)
                print(f"  Uploaded {Path(local).name}")
            except Exception as e:
                print(f"  Warning uploading {local}: {e}")


# ============================================================================
# 10. MAIN
# ============================================================================

def main() -> None:
    banner("STAGE 0 CLOSEOUT — T2-T7  (protocol: consolidated-flagship-112/1)")
    print(f"  OUT_DIR:   {OUT_DIR}")
    print(f"  CKPT_DIR:  {CKPT_DIR}")
    print(f"  VAL_ROOT:  {VAL_ROOT}")
    print(f"  EPS_PIXEL: {EPS_PIXEL:.6f}  ({EPS_PIXEL*255:.3f}/255)")

    # Data
    bootstrap_imagenet100(DATA_ROOT)

    # T2: Fetch + evaluate all arms
    banner("T2 CANONICAL HARNESS")
    arm_results: Dict[str, Any] = {}
    for arm in ARMS:
        ckpt_path = fetch_checkpoint(arm)
        result = run_t2_arm(arm, ckpt_path)
        arm_results[arm["arm_id"]] = result

    harness_sha = hashlib.sha256(
        (REPO_DIR / "evaluation" / "harness.py").read_bytes()
    ).hexdigest() if (REPO_DIR / "evaluation" / "harness.py").exists() else "N/A"

    harness = {
        "protocol_id": "consolidated-flagship-112/1",
        "prereg_file": "prereg/stage0_decision_v1.json",
        "harness_sha256": harness_sha,
        "eps_pixel": EPS_PIXEL,
        "eps_pixel_255": EPS_PIXEL * 255,
        "n_clean": 5000,
        "n_robust": 5000,
        "arms": arm_results,
    }
    harness_path = OUT_DIR / "stage0_harness_results.json"
    harness_path.write_text(json.dumps(harness, indent=2))
    print(f"\nT2 complete. Saved {harness_path}")

    # T3: Verdict
    verdict = run_t3_verdict(harness)

    # T4c: Slope
    try:
        slope = run_t4c_slope()
    except Exception as e:
        print(f"  T4c warning: {e}")
        slope = {}

    # T5-T7: Report
    report_path = run_t5t7_report(harness, verdict, slope)

    # T6: Upload
    upload_to_hf(report_path, harness)

    banner(f"DONE — Verdict: {verdict['verdict']}")
    elapsed = (time.time() - SESSION_T0) / 3600.0
    print(f"  Total wall time: {elapsed:.2f}h")


if __name__ == "__main__":
    main()


Installing torchattacks...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.7/178.7 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 4.6 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
datasets 4.8.5 requires requests>=2.32.2, but you have requests 2.25.1 which is incompatible.
nilearn 0.13.1 requires requests>=2.30.0, but you have requests 2.25.1 which is incompatible.
id 1.6.1 requires urllib3<3,>=2, but you have urllib3 1.26.20 which is incompatible.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.4.6 which is incompatible.
ydata-profiling 4.18.4 requires requests<3,>=2.32.0, but you have requests 2.25.1 which is incompatible.
docker 7.1.0 requires requests>=2.26.0, but you have requests 2.

HF_TOKEN authenticated (starts with: hf_jhD...)


Switched to a new branch 'stage0-closeout'


Branch 'stage0-closeout' set up to track remote branch 'stage0-closeout' from 'origin'.
HEAD is now at d81c4df T1 checkpoint intake manifest and intake script
Device: cuda
GPU: Tesla T4 (15.6 GB VRAM)

  STAGE 0 CLOSEOUT — T2-T7  (protocol: consolidated-flagship-112/1)
  OUT_DIR:   /kaggle/working/stage0_results
  CKPT_DIR:  /kaggle/working/checkpoints
  VAL_ROOT:  /kaggle/working/data/imagenet100/val
  EPS_PIXEL: 0.021243  (5.417/255)
source: clane9/imagenet-100@0519dc2f402a3a18c6e57f7913db059215eee25b (18 shards)
  data/train-00000-of-00017.parquet: 7453 rows -> train/ (16s)
  data/train-00001-of-00017.parquet: 7453 rows -> train/ (15s)
  data/train-00002-of-00017.parquet: 7453 rows -> train/ (15s)
  data/train-00003-of-00017.parquet: 7453 rows -> train/ (15s)
  data/train-00004-of-00017.parquet: 7453 rows -> train/ (15s)
  data/train-00005-of-00017.parquet: 7452 rows -> train/ (15s)
  data/train-00006-of-00017.parquet: 7452 rows -> train/ (16s)
  data/train-00007-of-00017.parquet: 7